
<font color = green >

## Home Task

</font>

Objective: Apply at least two (2) modern sentiment analysis methods to classify text data and evaluate their performance




Dataset:

Sentiment Analysis Dataset: https://www.cs.cornell.edu/people/pabo/movie-review-data/rt-polaritydata.tar.gz

alternative source:
[rt-polaritydata](https://github.com/dennybritz/cnn-text-classification-tf/tree/master/data/rt-polaritydata)

Each line in these two files corresponds to a single snippet (usually containing roughly one single sentence); all snippets are down-cased.

[More info about dataset](https://www.cs.cornell.edu/people/pabo/movie-review-data/rt-polaritydata.README.1.0.txt)


- rt-polarity.neg: Contains negative reviews.
- rt-polarity.pos: Contains positive reviews

### Task Description:

1. Data Loading & Preparation:

    - Load rt-polarity.neg and rt-polarity.pos.
    - Split each file into individual snippets.
    - Assign labels (0 for negative, 1 for positive).
    - Combine into a single dataset.
    - Split the dataset into training and testing sets.
2. Implement and evaluate at least three (3) methods from the lecture, prioritizing modern approaches.
3. For each implemented method, report and compare classification metrics: Accuracy, Precision, Recall, and F1-score.

In [1]:
import numpy as np
import pandas as pd

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import RegexpTokenizer
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split

nltk.download('stopwords')

import gensim
from gensim.models import Word2Vec
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments
from datasets import Dataset
import torch

from transformers import pipeline

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\koder/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
fn='rt-polarity.neg'

with open(fn, "r",encoding='utf-8', errors='ignore') as f: # some invalid symbols encountered
    content = f.read()
texts_neg=  content.splitlines()
print ('len of texts_neg = {:,}'.format (len(texts_neg)))
for review in texts_neg[:5]:
    print ( '\n', review)

len of texts_neg = 5,331

 simplistic , silly and tedious . 

 it's so laddish and juvenile , only teenage boys could possibly find it funny . 

 exploitative and largely devoid of the depth or sophistication that would make watching such a graphic treatment of the crimes bearable . 

 [garbus] discards the potential for pathological study , exhuming instead , the skewed melodrama of the circumstantial situation . 

 a visually flashy but narratively opaque and emotionally vapid exercise in style and mystification . 


In [3]:
fn='rt-polarity.pos'

with open(fn, "r",encoding='utf-8', errors='ignore') as f:
    content = f.read()
texts_pos=  content.splitlines()
print ('len of texts_pos = {:,}'.format (len(texts_pos)))
for review in texts_pos[:5]:
    print ('\n', review)

len of texts_pos = 5,331

 the rock is destined to be the 21st century's new " conan " and that he's going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal . 

 the gorgeously elaborate continuation of " the lord of the rings " trilogy is so huge that a column of words cannot adequately describe co-writer/director peter jackson's expanded vision of j . r . r . tolkien's middle-earth . 

 effective but too-tepid biopic

 if you sometimes like to go to the movies to have fun , wasabi is a good place to start . 

 emerges as something rare , an issue movie that's so honest and keenly observed that it doesn't feel like one . 


In [4]:
labels_neg = [0] * len(texts_neg)
labels_pos = [1] * len(texts_pos)

texts = texts_neg + texts_pos
labels = labels_neg + labels_pos

df = pd.DataFrame({'text': texts, 'label': labels})

train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['text'], df['label'], test_size=0.2, random_state=13, stratify=df['label']
)
df

,text,label
0,"simplistic , silly and tedious .",0
1,"it's so laddish and juvenile , only teenage bo...",0
2,exploitative and largely devoid of the depth o...,0
3,[garbus] discards the potential for pathologic...,0
4,a visually flashy but narratively opaque and e...,0
...,...,...
10657,both exuberantly romantic and serenely melanch...,1
10658,mazel tov to a film about a family's joyous li...,1
10659,standing in the shadows of motown is the best ...,1
10660,it's nice to see piscopo again after all these...,1


In [5]:
stop_words = set(stopwords.words('english'))
tokenizer = RegexpTokenizer(r'\w+')

def preprocess(text):     
    tokenized_text = tokenizer.tokenize(text.lower())
    return [w for w in  tokenized_text if w not in stop_words]

df['tokens'] = df['text'].apply(preprocess)
df['tokens']

0                             [simplistic, silly, tedious]
1        [laddish, juvenile, teenage, boys, could, poss...
2        [exploitative, largely, devoid, depth, sophist...
3        [garbus, discards, potential, pathological, st...
4        [visually, flashy, narratively, opaque, emotio...
                               ...                        
10657    [exuberantly, romantic, serenely, melancholy, ...
10658    [mazel, tov, film, family, joyous, life, actin...
10659    [standing, shadows, motown, best, kind, docume...
10660    [nice, see, piscopo, years, chaykin, headly, p...
10661    [provides, porthole, noble, trembling, incoher...
Name: tokens, Length: 10662, dtype: object

### Word2Vec

In [6]:
w2v_model = Word2Vec(df['tokens'], vector_size=200, window=5, min_count=2, workers=4, epochs = 30)

def get_vector(tokens):
    vectors = [w2v_model.wv[token] for token in tokens if token in w2v_model.wv]
    return np.mean(vectors, axis=0) if vectors else np.zeros(w2v_model.vector_size)

X_train_w2v = np.array([get_vector(tokens) for tokens in df.loc[train_texts.index, 'tokens']])
X_test_w2v = np.array([get_vector(tokens) for tokens in df.loc[test_texts.index, 'tokens']])

clf_w2v = LogisticRegression(max_iter=1000)
clf_w2v.fit(X_train_w2v, train_labels)
y_pred_w2v = clf_w2v.predict(X_test_w2v)

print("Word2Vec")
print(classification_report(test_labels, y_pred_w2v))

Word2Vec
              precision    recall  f1-score   support

           0       0.66      0.66      0.66      1067
           1       0.66      0.66      0.66      1066

    accuracy                           0.66      2133
   macro avg       0.66      0.66      0.66      2133
weighted avg       0.66      0.66      0.66      2133



### DistilBERT

In [7]:
model_name = "distilbert-base-uncased"
sample_size = 5000  
test_sample_size = 1000

train_indices = np.random.choice(len(train_texts), min(sample_size, len(train_texts)), replace=False)
X_train_sample = train_texts.iloc[train_indices].reset_index(drop=True)
y_train_sample = train_labels.iloc[train_indices].reset_index(drop=True)

test_indices = np.random.choice(len(test_texts), min(test_sample_size, len(test_texts)), replace=False)
X_test_sample = test_texts.iloc[test_indices].reset_index(drop=True)
y_test_sample = test_labels.iloc[test_indices].reset_index(drop=True)

train_dataset = Dataset.from_dict({
    'text': X_train_sample,
    'label': y_train_sample
})

test_dataset = Dataset.from_dict({
    'text': X_test_sample,
    'label': y_test_sample
})

In [8]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

tokenized_train = tokenized_train.remove_columns(["text"])
tokenized_test = tokenized_test.remove_columns(["text"])
tokenized_train.set_format("torch")
tokenized_test.set_format("torch")

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [9]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
)

trainer.train()

predictions = trainer.predict(tokenized_test)
pred_labels = np.argmax(predictions.predictions, axis=1)

print("DistilBERT :")
print(classification_report(y_test_sample, pred_labels))

Step,Training Loss
10,0.697500
20,0.712800
30,0.695400
40,0.686100
50,0.664700
60,0.638600
70,0.554500
80,0.484000
90,0.504500
100,0.406600


DistilBERT :
              precision    recall  f1-score   support

           0       0.84      0.82      0.83       509
           1       0.82      0.84      0.83       491

    accuracy                           0.83      1000
   macro avg       0.83      0.83      0.83      1000
weighted avg       0.83      0.83      0.83      1000



### Zero-shot

In [6]:
zero_shot_classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

candidate_labels = ["positive", "negative"]

y_pred = []
for text in test_texts:
    truncated_text = text[:512] 
    result = zero_shot_classifier(truncated_text, candidate_labels)
    predicted_label = result['labels'][0]
    y_pred.append(1 if predicted_label == "positive" else 0)


print("Zero-shot :")
print(classification_report(test_labels, y_pred, target_names=["negative", "positive"]))

Device set to use cuda:0
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Zero-shot :
              precision    recall  f1-score   support

    negative       0.75      0.90      0.82      1067
    positive       0.88      0.71      0.78      1066

    accuracy                           0.80      2133
   macro avg       0.81      0.80      0.80      2133
weighted avg       0.81      0.80      0.80      2133

